In [7]:
from gen_ai_hub.proxy.core.proxy_clients import set_proxy_version
from hana_ml import dataframe
from hana_ml.algorithms.pal.utility import DataSets

#use btp proxy llm
#

In [ ]:
from gen_ai_hub.proxy.langchain import init_llm
from hana_ai.agents.hana_dataframe_agent import create_hana_dataframe_agent
from hana_ai.vectorstore.embedding_service import GenAIHubEmbeddings
llm = init_llm('gpt-4', temperature=0.0, max_tokens=512)


In [ ]:
query = """I'm looking for a proper time series model for my data. My data's characteristics are below:
1. Intermittent Test: No
2. Seasonality Test: type is Multiplicative, period is 4, acf is 0.5
3. Stationarity Test: stationary is 0, kpss_stat is 0.26, p_value is 0.01, lags is 5, number of observations is 89, critical values are {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739}
4. Trend Test: upward trend.

Can you help me to find a proper time series model for my data?
"""
llm.invoke(query).content

In [ ]:
#connection to hana
#url, port, user = "hanagpu02.lenovo.wdf.sap.corp", 30015, "USER1"
url="hcp-ml-validate.hana-ml.c.ap-cn-1.cloud.sap"
user="MLAPITESTER"
passwd="<HANA_PASSWORD>5"
port=30315
connection_context = dataframe.ConnectionContext(url, port, user, passwd)
hana_df, _, _, _ = DataSets.load_boston_housing_data(connection_context)
hana_df = hana_df.to_head("ID")

<img src="dfagent.png" />

In [ ]:
from hana_ai.tools.toolkit import HANAMLToolkit
from hana_ai.vectorstore.hana_vector_engine import HANAMLinVectorEngine

toolkit = HANAMLToolkit()
hana_vec = HANAMLinVectorEngine(connection_context, "hana_vec_hana_ml_python_knowledge")
hana_vec.create_knowledge()
toolkit.set_vectordb(hana_vec)


In [46]:
knowledge_df = hana_vec.get_knowledge()

In [ ]:
knowledge_df.head(5).collect()

In [13]:
agent = create_hana_dataframe_agent(llm=llm, toolkit=toolkit, df=hana_df, verbose=True, handle_parsing_errors=True)

In [48]:
# query = """
# Question: create a dataset report.
# Code template: 
# from hana_ml.visualizers.unified_report import UnifiedReport
# UnifiedReport(<data_frame>).build(key=None).display()

# """

query = """
to use boston housing price table to create a dataset report.

"""

In [ ]:
agent.invoke(query)

In [ ]:
agent.run("create a dataset report.")

In [ ]:
agent.invoke("Create Automatic Regression model on this dataframe with max_eval_time_mins=10. Provide key is ID, background_size=100 and model_table_name='my_model' in the fit function and execute it. ")

In [ ]:
agent = create_hana_dataframe_agent(llm=llm, toolkit=toolkit, df=hana_df.deselect("MEDV").head(1), verbose=True)
agent.run("Do the predict on this dataframe with key 'ID' and model 'mymodel'. Save the predict result as the table name 'predcit_result'. Explain the predict result based on the reason code.")